# Deploying Deepseek-OCR with SageMaker SDK v3 and DJL LMI

This notebook demonstrates how to deploy the **Deepseek-OCR** multimodal model on Amazon SageMaker using the latest **SageMaker SDK v3** with **DJL Large Model Inference (LMI)** powered by the **vLLM** backend. This setup enables high-performance optical character recognition and document understanding at scale.

## What You'll Learn
- Deploy the Deepseek-OCR model using SageMaker SDK v3
- Configure DJL LMI with vLLM for optimized inference
- Extract structured data from images using Deepseek-OCR
- Handle concurrent requests with async processing
- Manage SageMaker endpoints and resources

### Prerequisites
Note: Ensure you have sagemaker and ipywidgets installed in your environment. The ipywidgets package is required to monitor endpoint deployment progress in Jupyter notebooks.

In [ ]:
!pip install sagemaker

In [ ]:
# Import required libraries
import json
import base64
from PIL import Image
import io
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core.helper.session_helper import Session

In [ ]:
role_arn = get_execution_role()
sess = Session()
region = sess._region_name

print(role_arn)

In [ ]:
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant
from sagemaker.core.resources import Model, EndpointConfig, Endpoint

# Set up environment variables for DJL LMI with vLLM backend
environment = {
    "HF_MODEL_ID": "deepseek-ai/DeepSeek-OCR",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_TENSOR_PARALLEL_DEGREE": "1",
    "OPTION_ENABLE_PREFIX_CACHING": "false",
    "OPTION_MM_PROCESSOR_CACHE_GB": "0",
    "OPTION_GPU_MEMORY_UTILIZATION": "0.88",
    "OPTION_MAX_ROLLING_BATCH_SIZE": "16",
    "MAX_CONCURRENT_REQUESTS": "32",
}

version = "v7"
# Use the DJL LMI container
container_definition = ContainerDefinition(
    image=f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.36.0-lmi19.0.0-cu128",
    environment=environment
)

# Create model and endpoint
model = Model.create(
    model_name=f"deepseek-ocr-{version}",
    primary_container=container_definition,
    execution_role_arn=role_arn,
    region=region
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=f"deepseek-ocr-config-{version}",
    production_variants=[
        ProductionVariant(
            variant_name="Primary",
            model_name=model.model_name,
            instance_type="ml.g6.8xlarge",
            initial_instance_count=1
        )
    ]
)

endpoint = Endpoint.create(
    endpoint_name=f"deepseek-ocr-vllm-{version}",
    endpoint_config_name=endpoint_config.endpoint_config_name
)

In [ ]:
endpoint.wait_for_status("InService")

In [ ]:
import json
import base64
from pprint import pprint
from PIL import Image
import io

image_1 = Image.open("img/schedule_table.png").convert("RGB")

# Convert to PNG bytes (not raw pixels)
image_bytes = io.BytesIO()
image_1.save(image_bytes, format='PNG')
image_base64 = base64.b64encode(image_bytes.getvalue()).decode('utf-8')

def invoke_ocr(prompt: str) -> str:
    request = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}}
                ]
            }
        ],
        "max_tokens": 2048,
        "temperature": 0.0,
        "stop": None
    }

    response_model = endpoint.invoke(
        json.dumps(request),
        content_type="application/json"
    )

    response = json.loads(response_model.body.read().decode('utf-8'))
    # pprint(response) # Uncomment for full output
    content = response['choices'][0]['message']['content']
    return content


In [20]:
from IPython.display import Markdown, display
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Invoke and display output as markdown
content = invoke_ocr("<image>\nFree OCR. ")
display(Markdown(content))

1. **Staff Shift Schedule**
   - West Sydneemouth Medical Center - Week of October 11, 2025

| Employee       | ID  | Department | Sat 11       | Sun 12       | Mon 13       | Tue 14       | Wed 15       | Thu 16       | Fri 17       |
|----------------|-----|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|
| Luz Davis     | KBPL7D | Surgery    | Morning 07:00-15:00 |              |              |              |              | Morning 07:00-15:00 |              |
| Felix Bergnaum| QHH6AM| General Ward | Leave Vacation | Leave Vacation |              |              |              |              |              |
| Judith Hagenes| YFTLIA| Neurology   | Afternoon 15:00-23:00 | Morning 07:00-15:00 |              | Morning 07:00-15:00 | Leave Sick Leave |              |              |
| Barbara Gislason| DSBOA4| Neurology   | Leave Personal |              | Leave Personal | Morning 07:00-15:00 | Leave Personal |              | Afternoon 15:00-23:00 |
| Andres Runte  | HAHGNZ| ICU        | Leave Personal |              | Afternoon 15:00-23:00 | Leave Training | Morning 07:00-15:00 |              |              |
| Jeffrey Littel | HWM4MU| Emergency   | Afternoon 15:00-23:00 | Leave Training |              | Afternoon 15:00-23:00 | Leave Training | Leave Training | Night 23:00-07:00 |
| James Romaguera| ZGG090| Pediatrics  |              |              | Leave Training | Morning 07:00-15:00 |              | Morning 07:00-15:00 |              |
| Harriet Smith  | IYIOKG| General Ward | Leave Personal |              | Afternoon 15:00-23:00 |              | Morning 07:00-15:00 | Night 23:00-07:00 | Afternoon 15:00-23:00 |
| Robin Hodkiewicz| 6SFHOR| General Ward |              | Afternoon 15:00-23:00 |              | Morning 07:00-15:00 |              |              |              |
| Theresa Wolf   | AFURZA| Surgery    |              | Morning 07:00-15:00 | Afternoon 15:00-23:00 |              |              | Night 23:00-07:00 | Afternoon 15:00-23:00 |
| Vicky D'Amore | RURDPT| General Ward | Afternoon 15:00-23:00 | Morning 07:00-15:00 |              | Afternoon 15:00-23:00 |              | Afternoon 15:00-23:00 |              |

In [23]:
from IPython.display import Pretty
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Invoke and display output as markdown
content = invoke_ocr("<image>\n<|grounding|>What week is the schedule for?")
display(Pretty(content))

The schedule is for the week of October 11, 2025.

## Compare output to input image
![img](./img/schedule_table.png)

## Step 7: Clean Up Resources

Clean up all created resources.

In [ ]:
# Clean up resources
endpoint_config = EndpointConfig.get(endpoint_config_name=endpoint.endpoint_name)

model.delete()
endpoint.delete()
endpoint_config.delete()

print("All resources successfully deleted!")

## Summary

This notebook walked through a complete end-to-end workflow for deploying Deepseek-OCR on SageMaker:

1. **Environment Setup**: Configured DJL LMI with vLLM backend parameters including tensor parallelism, GPU memory utilization, and async mode
2. **Model Deployment**: Created a SageMaker model, endpoint configuration, and deployed to a GPU instance
3. **Inference Testing**: Demonstrated OCR capabilities by extracting structured data (staff schedules) from images with high accuracy
4. **Multi-turn Queries**: Showed how to ask specific questions about extracted content using grounding
5. **Resource Cleanup**: Properly tore down all created resources

The result is a production-ready OCR service capable of processing images and returning structured, actionable data—perfect for document automation, data extraction, and intelligent document processing workflows.